# MEC 2026 - Online Music Alignment with Matchmaker (Audio)

**Music Alignment Workshop, MEC 2026 — Section 3 (Online Alignment), Part 1 (Audio)**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pymatchmaker/mec2026_alignment_workshop/blob/online-alignment/notebooks/online_alignment_audio.ipynb)

This notebook accompanies the online alignment section of **Music Alignment Uncovered: Representations, Algorithms and Hands-On Tools** at MEC 2026.

In this notebook we align **one audio recording and the corresponding score** using the [Matchmaker](https://github.com/pymatchmaker/matchmaker). 

## 0. Setup

Run the setup cells once at the beginning. If you already installed the dependencies, you may skip the installation cells.

Workshop resources live under `mec-variation/` at the repo root. Make sure that directory is present alongside this notebook (clone the workshop repo if you haven't).

For audio score following, Matchmaker may need FluidSynth and libsndfile. In Colab the apt cell below installs them; locally install them via your system package manager, e.g. `conda install -c conda-forge fluidsynth libsndfile`.

In [ ]:
#@title Setup dependencies (apt + pip + clone) { display-mode: "form" }
import importlib.util
import shutil
import subprocess
from pathlib import Path

try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False

if IN_COLAB and shutil.which("apt-get") is not None:
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(
        ["apt-get", "install", "-y", "-qq", "fluidsynth", "libsndfile1", "portaudio19-dev"],
        check=True,
    )
else:
    print("Skipping apt-get setup outside Colab.")

%pip install -q "setuptools>=80,<81" "numpy>=1.26.3,<2.0" "pymatchmaker>=0.3.0" pandas matplotlib librosa partitura ipython

if IN_COLAB and not Path("/content/mec2026_alignment_workshop").exists():
    !git clone --depth 1 --branch online-alignment https://github.com/pymatchmaker/mec2026_alignment_workshop.git

if IN_COLAB:
    print("\nRestarting runtime so the downgraded numpy is loaded cleanly...")
    import IPython
    IPython.Application.instance().kernel.do_shutdown(True)


In [ ]:
#@title Imports and data paths { display-mode: "form" }
import json
import warnings

# Silence noisy deprecation/complex warnings from matchmaker's dependencies.
warnings.filterwarnings("ignore")

import librosa
import matchmaker
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import partitura as pt
from IPython.display import Audio, Image, display
from pathlib import Path
from matchmaker import Matchmaker
from matchmaker.features.audio import (
    ChromagramProcessor,
    CQTProcessor,
    CQTSpectralFluxProcessor,
)
from matchmaker.features.midi import onset_pianoroll
from matchmaker.matchmaker import AVAILABLE_METHODS, DEFAULT_METHOD, DEFAULT_PROCESSOR
from partitura.musicanalysis.performance_codec import get_time_maps_from_alignment

print("matchmaker", matchmaker.__version__)

# Locate mec-variation/: try local (notebook in repo) first, fall back to the
# Colab clone path created by the setup cell.
DATA_DIR = Path("../mec-variation")
if not DATA_DIR.exists():
    DATA_DIR = Path("/content/mec2026_alignment_workshop/mec-variation")
SCORE_FILE = DATA_DIR / "scores/musicxml/short.musicxml"
AUDIO_FILE = DATA_DIR / "performances/wav/short1.wav"
MIDI_FILE = DATA_DIR / "performances/midi/short1.mid"
MATCH_FILE = DATA_DIR / "alignments/match/short1.match"
SNIPPET_FILE = DATA_DIR / "performances/wav/snippet.wav"
PREVIEW_IMAGE = DATA_DIR / "scores/simple_mozart_first_two_measures.png"
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

perf, alignment, score = pt.load_match(filename=str(MATCH_FILE), create_score=True)
snote_array = score.note_array()
_, stime_to_ptime_map = get_time_maps_from_alignment(
    perf.note_array(), snote_array, alignment, onset_aggregation_fun=np.min,
)
NOTE_ANNOTATIONS = stime_to_ptime_map(np.unique(snote_array["onset_beat"]))

print("Data directory:", DATA_DIR.resolve())
for path in [SCORE_FILE, AUDIO_FILE, MIDI_FILE, MATCH_FILE, SNIPPET_FILE, PREVIEW_IMAGE]:
    print(path.name, "ok" if path.exists() else "missing")
print(f"NOTE_ANNOTATIONS: {len(NOTE_ANNOTATIONS)} entries, {NOTE_ANNOTATIONS.min():.3f}s ~ {NOTE_ANNOTATIONS.max():.3f}s")

The recording below is the audio we will align against the score throughout the rest of this notebook (`short1.wav`, a take of the first 24 measures of Mozart K.265 var. 1). Every method we run from here on uses this same file as the performance input.

In [ ]:
display(Audio(filename=str(AUDIO_FILE)))

## 1. Difference between online vs. offline alignment

**Offline alignment** receives the complete performance before it computes the alignment. It can revise earlier decisions because it can see the future.

**Online alignment** receives a stream of observations. At each audio frame or MIDI event, it must estimate the current score position using only the past and present. The next estimate may step back to an earlier score position, but each estimate, once emitted, is final. Past outputs cannot be revised.

Technical constraints:

- Causality: no access to future observations.
- Latency: real-time systems must finish the computation within single frame budget.

Online-specific challenges (vs offline)

- Ambiguity: repeated or harmonically similar passages can look alike locally.
- Drift & no recovery: locally-confused frames can't be revised, so errors compound over long pieces.
- Abrupt tempo changes: hard to catch up since a global cost search (which offline DTW would use) is not available.
- Noisy input: even during musically silent passages, the audio signal is not literally zero. Microphone hiss, room tone, breath, and non-instrument sounds all leak in and can confuse the feature stream.

Audio-specific challenges (vs MIDI)

- Polyphonic ambiguity: simultaneously-sounding notes share spectral bins, so per-frame pitch is uncertain.
- Onset uncertainty: onsets are smeared across the analysis window rather than landing on discrete event times. To compensate, audio-based score followers often design features to be onset-aware (e.g. by mixing harmonic features with a spectral-flux or onset-strength signal).
- Timbre & recording variation: instrument, microphone, room change the spectrogram; score-synthesized templates never match the performance exactly.

## 2. Audio Features

Matchmaker doesn't feed raw audio samples into the score follower. The `AudioStream` slices the source into fixed-size hops, and each hop goes through a `Processor` that returns one feature vector.

Two processors used in this workshop:

- `ChromagramProcessor` — 12-bin chroma. Default for audio, used by the Arzt OLTW method.
- `CQTProcessor` — 88-bin constant-Q transform. Used by the OuterHMM method.

Below we run these processors hop-by-hop on a short snippet and render the feature image one column at a time.

In [ ]:
from IPython.display import clear_output, Audio
import matplotlib.image as mpimg

print(f"Below is the short score snippet for feature visualization, along with the audio snippet (first 2 measures).")
display(Image(filename=str(PREVIEW_IMAGE), width=700))
display(Audio(filename=str(SNIPPET_FILE)))

SCORE_IMG = mpimg.imread(str(PREVIEW_IMAGE))
SCORE_AR = SCORE_IMG.shape[1] / SCORE_IMG.shape[0]


def live_stream_processor(processor, sample_rate, hop_length, title, ylabel, yticks, yticklabels):
    audio_y, _ = librosa.load(str(SNIPPET_FILE), sr=sample_rate, mono=True)
    n_fft = getattr(processor, "n_fft", 2 * hop_length)
    features = np.array([
        np.asarray(processor((audio_y[s:s + n_fft], s / sample_rate))[0]).ravel()
        for s in range(0, len(audio_y) - n_fft + 1, hop_length)
    ]).T  # (feature_dim, n_frames)
    processor.reset()

    n_frames = features.shape[1]
    duration = n_frames * hop_length / sample_rate
    step = max(1, n_frames // 12)
    fig, (sax, ax) = plt.subplots(
        2, 1, figsize=(SCORE_AR * 2.0, 4.0), gridspec_kw={"height_ratios": [1, 1]},
    )
    sax.imshow(SCORE_IMG); sax.set_axis_off(); sax.set_title("Score (first two measures)")
    for k in range(step, n_frames + 1, step):
        ax.clear()
        ax.imshow(features[:, :k], origin="lower", aspect="auto", interpolation="nearest", vmin=0,
                  extent=[0, k * hop_length / sample_rate, 0, features.shape[0]])
        ax.set_xlim(0, duration)
        ax.set(xlabel="Performance time (s)", ylabel=ylabel, title=title)
        ax.set_yticks(yticks); ax.set_yticklabels(yticklabels); ax.grid(False)
        clear_output(wait=True); display(fig)
    plt.close(fig)
    return features

### 2-1: Chromagram

`ChromagramProcessor` collapses spectral energy into 12 pitch classes (C, C#, …, B), dropping octave information. Each call takes one hop of audio and returns a 12-dim row.

In [ ]:
chroma_sr = 44100
chroma_hop = chroma_sr // 30  # 50 fps
chroma_processor = ChromagramProcessor(sample_rate=chroma_sr, hop_length=chroma_hop)
print("Processor:", chroma_processor.__class__.__name__)
print(f"  sample_rate = {chroma_sr} Hz, hop_length = {chroma_hop} samples")
print(f"  one feature row every {chroma_hop / chroma_sr * 1000:.1f} ms")

pitch_class_names = ["C", "C#", "D", "D#", "E", "F", "F#", "G", "G#", "A", "A#", "B"]
pitch_yticks = np.arange(12) + 0.5

chroma_buffer = live_stream_processor(
    chroma_processor, chroma_sr, chroma_hop,
    title="ChromagramProcessor (streamed from audio)",
    ylabel="Pitch class",
    yticks=pitch_yticks,
    yticklabels=pitch_class_names,
)
print(f"Final feature shape: {chroma_buffer.shape} = pitch classes x hops")

# Online vs. offline sanity check: feed the whole snippet through in one shot.
chroma_processor.reset()
full_audio, _ = librosa.load(str(SNIPPET_FILE), sr=chroma_sr, mono=True)
offline_chroma = chroma_processor((full_audio, 0.0))[0].T  # (12, n_frames)
m = min(chroma_buffer.shape[1], offline_chroma.shape[1])
print(f"\nOffline chroma shape: {offline_chroma.shape}")
print(f"Streamed (online) chroma shape: {chroma_buffer.shape}")
print(f"Max |online - offline| over {m} overlapping frames: {np.abs(chroma_buffer[:, :m] - offline_chroma[:, :m]).max():.2e}")

fig, ax = plt.subplots(figsize=(SCORE_AR * 2.0, 2.0))
ax.imshow(
    offline_chroma, origin="lower", aspect="auto", interpolation="nearest",
    extent=[0, offline_chroma.shape[1] * chroma_hop / chroma_sr, 0, 12], vmin=0,
)
ax.set(xlabel="Performance time (s)", ylabel="Pitch class",
       title="ChromagramProcessor (offline: full audio in one call)")
ax.set_yticks(pitch_yticks)
ax.set_yticklabels(pitch_class_names)
plt.show()

### 2-2: CQT (Constant-Q Transform)

`CQTProcessor` returns an 88-bin constant-Q transform per hop, one bin per piano key (A0–C8). Unlike chroma, the octave is preserved — you can read off actual notes rather than just pitch classes.

The audio `outerhmm` follower uses the related `CQTSpectralFluxProcessor`, which packs the same 88 CQT bins plus a 89th spectral-flux row used as an exit-boost signal. For visualization we use the plain CQT version since the flux value lives on a very different scale (sum over 88 bins) and would dominate a shared colormap.

In [ ]:
cqt_sr = 16000
cqt_hop = cqt_sr // 25  # 25 fps
cqt_processor = CQTProcessor(sample_rate=cqt_sr, hop_length=cqt_hop)
print("Processor:", cqt_processor.__class__.__name__)
print(f"  sample_rate = {cqt_sr} Hz, hop_length = {cqt_hop} samples")
print(f"  one feature row every {cqt_hop / cqt_sr * 1000:.1f} ms")

cqt_yticks = [0, 24, 48, 72]
cqt_yticklabels = ["A0", "A2", "A4", "A6"]

cqt_buffer = live_stream_processor(
    cqt_processor, cqt_sr, cqt_hop,
    title="CQT (streamed from audio)",
    ylabel="Piano key (A0-C8)",
    yticks=cqt_yticks,
    yticklabels=cqt_yticklabels,
)
print(f"Final feature shape: {cqt_buffer.shape} = CQT bins x hops")

# Online vs. offline sanity check.
cqt_processor.reset()
full_audio_cqt, _ = librosa.load(str(SNIPPET_FILE), sr=cqt_sr, mono=True)
offline_cqt = cqt_processor((full_audio_cqt, 0.0))[0].T  # (88, n_frames)
m = min(cqt_buffer.shape[1], offline_cqt.shape[1])
print(f"\nOffline CQT shape: {offline_cqt.shape}")
print(f"Streamed (online) CQT shape: {cqt_buffer.shape}")
print(f"Max |online - offline| over {m} overlapping frames: {np.abs(cqt_buffer[:, :m] - offline_cqt[:, :m]).max():.2e}")

fig, ax = plt.subplots(figsize=(SCORE_AR * 2.0, 2.0))
ax.imshow(
    offline_cqt, origin="lower", aspect="auto", interpolation="nearest",
    extent=[0, offline_cqt.shape[1] * cqt_hop / cqt_sr, 0, offline_cqt.shape[0]],
    vmin=0,
)
ax.set(xlabel="Performance time (s)", ylabel="Piano key (A0-C8)",
       title="CQT (offline: full audio in one call)")
ax.set_yticks(cqt_yticks)
ax.set_yticklabels(cqt_yticklabels)
plt.show()

## 3. Matchmaker overview


The Matchmaker pipeline keeps the same shape for both input types; the differences above are absorbed by the choice of `Stream` and `Processor`:

```text
input source -> Stream        -> Processor -> OnlineAlignment -> current score position
(audio/MIDI)   (file or live)   (features)   (alignment method)  (beat position)
```

For this workshop, the main object is `Matchmaker`. `Matchmaker` class is an all-in-one class that encapsulates the entire alignment process, including `Processor` and `OnlineAlignment`. We use it in **simulation mode**: the input is a recorded audio or MIDI performance file, but Matchmaker feeds it through the same streaming pipeline used for real-time score following.

The cell below quickly runs Matchmaker on our example file and yields one estimated score position (in beats) per audio frame. After the loop ends, `mm.score_follower.alignment_path` stores a `(2, T)` array:

- row 0: estimated score beat
- row 1: performance time in seconds

Then, let's run the matchmaker in the simpliest way possible.

In [ ]:
mm = Matchmaker(
    score_file=SCORE_FILE,
    performance_file=AUDIO_FILE,
)

positions = []
for p in mm.run():
    print(f"Estimated score beat: {p:.3f}")
    positions.append(float(p))
    
print(f"Tracked {len(positions)} observations")
print(f"First 5 estimated score beats: {[round(p, 3) for p in positions[:5]]}")
print(f"Alignment path shape: {mm.score_follower.alignment_path.shape}")

The cell above used the minimum required arguments: just a score and a performance file. Matchmaker filled in the defaults for `method` and `processor`, detecting the input type.

Swapping in a different alignment algorithm is as simple as adding a `method="..."` argument to the constructor. The cell below lists which methods are available for each input type.

In [ ]:
print("Available methods:")
for input_type, methods in AVAILABLE_METHODS.items():
    print(f"  {input_type}: {methods}")

print("\nDefaults:")
print("  method:", DEFAULT_METHOD)
print("  processor:", DEFAULT_PROCESSOR)

### 3.1 Reading Matchmaker's score positions

Matchmaker reports the current position in **score beats**. For this Mozart example, the score positions come from Partitura's `note_array()` representation, especially the `onset_beat` field.

The unique note-onset beats become the main discrete score states used by several online followers.

In [ ]:
score = pt.load_musicxml(str(SCORE_FILE))
score_part = score[0]
note_array = score_part.note_array()
score_positions = np.unique(note_array["onset_beat"])

print(f"Number of notes: {len(note_array)}")
print(f"Number of unique score onset positions: {len(score_positions)}")
print("First 8 score positions in beats (first measure):")
print(score_positions[:8])

display(Image(filename=str(PREVIEW_IMAGE), width=700))

pd.DataFrame(note_array[:10])[["id", "pitch", "onset_beat", "duration_beat"]]

## 4. Running audio score following with two different alignment methods

Audio methods available in Matchmaker include:

- `"dixon"`: On-line time warping score follower (Dixon, 2005)
- `"arzt"`: Step-size-constrained on-line time warping score follower (Arzt et al., 2010, 2015)
- `"outerhmm"`: Outer-product HMM score follower  (Nakamura et al., 2014)
- `"skf"`: Switching Kalman Filter with hidden tempo (Jiang et al., 2020)
- `"pf"`: Particle filter score follower (Duan et al., 2011)

We will run two of these methods on our Mozart example: `"arzt"` and `"outerhmm"`, each representing a different alignment algorithm.

### 4.1 Dynamic programming

Online Time Warping (OLTW) incrementally aligns a live incoming performance to a reference sequence by updating a dynamic-programming warping path as each new audio or symbolic feature frame arrives.
Unlike an HMM-based follower, which treats score position as a hidden state and uses transition/emission probabilities, OLTW is a distance-based dynamic-programming method that directly searches for a low-cost path through the reference-performance similarity matrix in real time.

Here, the reference features are synthesized audio features from the score, using a MIDI synthesizer (FluidSynth) using piano soundfont. The performance features are chroma features extracted from the live audio.

In [ ]:
print(f"Performance file: {AUDIO_FILE.name}, with input mode: audio")
print(f"Running Matchmaker with score file ({SCORE_FILE.name}) and method 'arzt'...")

audio_arzt_mm = Matchmaker(
    score_file=SCORE_FILE,
    performance_file=AUDIO_FILE,
    input_type="audio",
    method="arzt",
)

audio_arzt_positions = [float(p) for p in audio_arzt_mm.run(verbose=False)]
audio_arzt_path = audio_arzt_mm.score_follower.alignment_path

print("Tracked audio observations:", len(audio_arzt_positions))

audio_arzt_eval = audio_arzt_mm.run_evaluation(
    perf_annotations=NOTE_ANNOTATIONS,
    debug=True,
    save_dir=RESULTS_DIR,
    run_name="audio_arzt",
    plot_dist_matrix=True,
)

print("Evaluation result:")
print(json.dumps(audio_arzt_eval, indent=2))

wp = np.loadtxt(RESULTS_DIR / "wp_audio_arzt.tsv", delimiter="\t")
gt = np.loadtxt(RESULTS_DIR / "gt_audio_arzt.tsv", delimiter="\t")

fig, ax = plt.subplots(figsize=(10, 10))
ax.scatter(wp[:, 1], wp[:, 0], color="blue", s=3, alpha=0.5, label="warping path")
ax.scatter(gt[:, 1], gt[:, 0], color="red", marker="x", s=15, zorder=3, label="ground truth")
ax.set(xlabel="performance time (s)", ylabel="score position (beats)", title="OLTWArzt alignment plot")
ax.grid(alpha=0.2)
ax.legend()
plt.show()

### 4.2 HMM audio input features: CQT + spectral flux

An HMM-based score follower models the current score position as a hidden state and infers it from incoming performance features using emission probabilities and transition probabilities between score positions. Compared with OLTW, it is more explicitly probabilistic: the system does not just search for a low-cost path, but maintains beliefs over possible score positions and updates them as new audio or MIDI observations arrive.

The audio `outerhmm` configuration uses `processor="cqt_spectral_flux"` for the incoming performance audio. Compared with chroma, this representation keeps a wider pitch range and adds a spectral-flux component that emphasizes frame-to-frame changes.

In [ ]:
print(f"Performance file: {AUDIO_FILE.name}, with input mode: audio")
print(f"Running Matchmaker with score file ({SCORE_FILE.name}) and method 'outerhmm'...")

audio_outerhmm_mm = Matchmaker(
    score_file=SCORE_FILE,
    performance_file=AUDIO_FILE,
    input_type="audio",
    method="outerhmm",
)

audio_outerhmm_positions = [p for p in audio_outerhmm_mm.run(verbose=False)]
audio_outerhmm_path = audio_outerhmm_mm.score_follower.alignment_path

print("Tracked audio observations:", len(audio_outerhmm_positions))
print("OuterHMM alignment path shape:", audio_outerhmm_path.shape)

audio_outerhmm_eval = audio_outerhmm_mm.run_evaluation(
    perf_annotations=NOTE_ANNOTATIONS,
    debug=True,
    save_dir=RESULTS_DIR,
    run_name="audio_outerhmm",
    plot_dist_matrix=True,
)

print("Evaluation result:")
print(json.dumps(audio_outerhmm_eval, indent=2))

wp = np.loadtxt(RESULTS_DIR / "wp_audio_outerhmm.tsv", delimiter="\t")
gt = np.loadtxt(RESULTS_DIR / "gt_audio_outerhmm.tsv", delimiter="\t")

fig, ax = plt.subplots(figsize=(10, 10))
ax.scatter(wp[:, 1], wp[:, 0], color="blue", s=3, alpha=0.5, label="warping path")
ax.scatter(
    gt[:, 1], gt[:, 0], color="red", marker="x", s=15, zorder=3, label="ground truth"
)
ax.set(xlabel="performance time (s)", ylabel="score position (beats)", title="OuterHMM alignment plot")
ax.grid(alpha=0.2)
ax.legend()
plt.show()

### 4.3 Audio evaluation and method comparison

Matchmaker's `run_evaluation` compares the follower's alignment path with the ground-truth annotations (built in section 0 from the `.match` file). It reports the result in two domains:

- **Beat-level** (score domain): predict the score beat at each ground-truth performance onset and measure the error in beats.
- **Ms-level** (performance domain): predict the performance time at each ground-truth score beat and measure the error in milliseconds.

For each domain we look at two kinds of numbers:

- **`*.mean (↓)`**: mean absolute error (lower is better). Units are beats for the score domain, milliseconds for the performance domain.
- **`*.<tolerance> (%, ↑)`**: percentage of estimates within the given tolerance, similar to the misalignment-rate convention used in offline sync evaluation (e.g. SyncToolbox ISMIR 2021 LBD notebook, which reports `% above threshold` for thresholds 30 / 50 / 100 / 150 / 200 / 300 / 500 / 1000 ms). For example, `beat.0.5b = 97.0%` means 97 % of frames land within 0.5 beats of the ground truth.

Two extra signals matter for **online** score following and Matchmaker keeps them in the raw eval dict (not in the summary table):

- **`rtf`** (real-time factor): processing time / performance duration. Below 1.0 means the follower keeps up with real-time playback.
- **`f_avg_latency`, `i_avg_latency`**: average frame-level / inter-frame latency in milliseconds, capturing the delay between observation arrival and position output.

In [ ]:
# Side-by-side metrics: arzt vs outerhmm.
# Display the full evaluation dictionaries without filtering columns.
# Keep the metric order readable: summary stats first, then tolerance hit rates.
summary_keys = ["mean", "median", "std", "skewness", "kurtosis"]

def flatten_eval_in_order(ev):
    row = {}
    for group in ["beat", "ms"]:
        metrics = ev.get(group, {})
        ordered_keys = [k for k in summary_keys if k in metrics]
        ordered_keys += [k for k in metrics.keys() if k not in ordered_keys]
        for key in ordered_keys:
            row[f"{group}.{key}"] = metrics[key]
    for group, metrics in ev.items():
        if group in ["beat", "ms"]:
            continue
        if isinstance(metrics, dict):
            for key, value in metrics.items():
                row[f"{group}.{key}"] = value
        else:
            row[group] = metrics
    return row

audio_eval_rows = []
for method, ev in [
    ("arzt + chroma", audio_arzt_eval),
    ("outerhmm + cqt_spectral_flux", audio_outerhmm_eval),
]:
    audio_eval_rows.append({"method": method, **flatten_eval_in_order(ev)})

audio_eval_df = pd.DataFrame(audio_eval_rows)
with pd.option_context("display.max_columns", None, "display.width", None):
    display(audio_eval_df)

# Tolerance hit-rate bar chart: beat and ms side by side (y-axis in %).
beat_tols = ["0.05b", "0.1b", "0.3b", "0.5b", "1b", "2b"]
ms_tols = ["50ms", "100ms", "300ms", "500ms", "1000ms", "2000ms"]
width = 0.35
fig, (ax_b, ax_m) = plt.subplots(1, 2, figsize=(14, 5))
for ax, tols, group in [(ax_b, beat_tols, "beat"), (ax_m, ms_tols, "ms")]:
    x = np.arange(len(tols))
    ax.bar(x - width / 2, [audio_arzt_eval[group][t] * 100 for t in tols], width, label="arzt", color="C0")
    ax.bar(x + width / 2, [audio_outerhmm_eval[group][t] * 100 for t in tols], width, label="outerhmm", color="C1")
    ax.set_xticks(x)
    ax.set_xticklabels(tols)
    ax.set(xlabel=f"{group} tolerance", ylabel="hit rate (%)", ylim=(0, 105),
           title=f"{group.capitalize()} tolerance hit rates")
    ax.legend()
    ax.grid(alpha=0.2, axis="y")
plt.tight_layout()
plt.show()

Arzt/OLTW often performs better on millisecond error because it directly optimizes a frame-level alignment path, making it very sensitive to local timing. OuterHMM is probabilistic and more state-based, which can be more robust, but its transitions and state discretization may smooth or delay the estimated position, leading to larger timing errors.

## 5. Tempo variation

So far we aligned `short1.wav` to the score. Let's see how the OuterHMM follower handles a slower take of the same passage: `short_slow1.wav`. The score, method, and reference features stay the same; only the performance changes.

The cell below rebuilds the ground truth annotations from the corresponding `.match` file (same procedure as in the setup cell), plots the tempo curves of the two takes, runs OuterHMM on the slow take, and compares the metrics with the original take.

In [ ]:
SLOW_AUDIO = DATA_DIR / "performances/wav/short_slow1.wav"
SLOW_MATCH = DATA_DIR / "alignments/match/short_slow1.match"

# Build slow-take annotations the same way as in the setup cell.
slow_perf, slow_alignment, slow_score = pt.load_match(filename=str(SLOW_MATCH), create_score=True)
slow_snote = slow_score.note_array()
_, slow_stime_to_ptime = get_time_maps_from_alignment(
    slow_perf.note_array(), slow_snote, slow_alignment, onset_aggregation_fun=np.min,
)
SLOW_NOTE_ANNOTATIONS = slow_stime_to_ptime(np.unique(slow_snote["onset_beat"]))
print(f"Slow performance has {len(SLOW_NOTE_ANNOTATIONS)} unique onsets, ends at {SLOW_NOTE_ANNOTATIONS.max():.2f}s")
display(Audio(filename=str(SLOW_AUDIO)))

# Tempo curve: local BPM between consecutive integer beats (downbeats).
# Interpolate perf time at each integer score beat so 16th-note micro-timing
# averages out.
from scipy.interpolate import interp1d
orig_beats = np.unique(snote_array["onset_beat"])
slow_beats = np.unique(slow_snote["onset_beat"])
int_beats = np.arange(int(np.ceil(orig_beats.min())), int(np.floor(orig_beats.max())) + 1)
orig_sec_at_int = interp1d(orig_beats, NOTE_ANNOTATIONS)(int_beats)
slow_sec_at_int = interp1d(slow_beats, SLOW_NOTE_ANNOTATIONS)(int_beats)
orig_bpm = 1.0 / np.diff(orig_sec_at_int) * 60
slow_bpm = 1.0 / np.diff(slow_sec_at_int) * 60
beat_mid = (int_beats[:-1] + int_beats[1:]) / 2

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(beat_mid, orig_bpm, color="C0", linewidth=1.5, alpha=0.8,
        label=f"short1 (median {np.median(orig_bpm):.0f} BPM)")
ax.plot(beat_mid, slow_bpm, color="C1", linewidth=1.5, alpha=0.8,
        label=f"short_slow1 (median {np.median(slow_bpm):.0f} BPM)")
ax.set(xlabel="score beat", ylabel="tempo (BPM)", title="Tempo curve: original vs slow take")
ax.grid(alpha=0.2)
ax.legend()
plt.show()

# Run matchmaker on the slow take.
slow_outerhmm_mm = Matchmaker(
    score_file=SCORE_FILE, performance_file=SLOW_AUDIO, input_type="audio", method="outerhmm",
)
slow_outerhmm_positions = [float(p) for p in slow_outerhmm_mm.run(verbose=False)]
slow_outerhmm_eval = slow_outerhmm_mm.run_evaluation(
    perf_annotations=SLOW_NOTE_ANNOTATIONS, debug=True, save_dir=RESULTS_DIR, run_name="audio_outerhmm_slow",
)

# Warping-path plots: original (left) vs slow (right) for outerhmm,
# with subplot widths proportional to each take's performance duration.
orig_duration = NOTE_ANNOTATIONS.max()
slow_duration = SLOW_NOTE_ANNOTATIONS.max()
width_per_sec = 0.3

fig, axes = plt.subplots(
    1, 2, figsize=(width_per_sec * (orig_duration + slow_duration), 6), sharey=True,
    gridspec_kw={"width_ratios": [orig_duration, slow_duration]},
)
for ax, take_label, wp_file, gt_x, gt_y in [
    (axes[0], "short1 (original)", "wp_audio_outerhmm.tsv", NOTE_ANNOTATIONS, orig_beats),
    (axes[1], "short_slow1", "wp_audio_outerhmm_slow.tsv", SLOW_NOTE_ANNOTATIONS, slow_beats),
]:
    wp = np.loadtxt(RESULTS_DIR / wp_file, delimiter="\t")
    ax.scatter(wp[:, 1], wp[:, 0], color="blue", s=5, alpha=0.7, label="warping path")
    ax.scatter(gt_x, gt_y, color="red", marker="x", s=15, zorder=3, label="ground truth")
    ax.set(xlabel="performance time (s)", title=f"outerhmm — {take_label}")
    ax.grid(alpha=0.2)
    ax.legend(loc="lower right")
axes[0].set_ylabel("score position (beats)")
plt.tight_layout()
plt.show()

# Side-by-side metrics: original vs slow take (outerhmm).
# Arrow markers indicate direction: ↓ smaller-is-better, ↑ larger-is-better.
# Tolerance hit rates are shown as percentages.
rows = []
for take, ev in [("original", audio_outerhmm_eval), ("slow", slow_outerhmm_eval)]:
    rows.append({
        "take": take,
        "beat.mean (↓)": ev["beat"]["mean"],
        "beat.0.3b (%, ↑)": ev["beat"]["0.3b"] * 100,
        "beat.0.5b (%, ↑)": ev["beat"]["0.5b"] * 100,
        "beat.1.0b (%, ↑)": ev["beat"]["1b"] * 100,
        "ms.mean (↓)": ev["ms"]["mean"],
        "ms.100ms (%, ↑)": ev["ms"]["100ms"] * 100,
        "ms.300ms (%, ↑)": ev["ms"]["300ms"] * 100,
        "ms.500ms (%, ↑)": ev["ms"]["500ms"] * 100,
    })
display(pd.DataFrame(rows))